In [0]:
%pip install openpyxl "pymongo[srv]" certifi

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./00_setup_mongodb

In [0]:
dbutils.widgets.text(
    "arquivo_entrada",
    "/Volumes/workspace/default/chamados-suporte/chamados_suporte_ficticios.xlsx",
    "Caminho do Excel de chamados"
)

dbutils.widgets.text(
    "mongo_database",
    "databricks_learning",
    "Database MongoDB"
)

dbutils.widgets.text(
    "mongo_collection",
    "chamados_raw",
    "Collection MongoDB"
)

dbutils.widgets.dropdown(
    "modo_carga",
    "upsert",
    ["upsert", "replace_collection"],
    "Modo de carga"
)

arquivo_entrada = dbutils.widgets.get("arquivo_entrada").strip()
mongo_database = dbutils.widgets.get("mongo_database").strip()
mongo_collection = dbutils.widgets.get("mongo_collection").strip()
modo_carga = dbutils.widgets.get("modo_carga").strip()

print("Arquivo de entrada:", arquivo_entrada)
print("Mongo database:", mongo_database)
print("Mongo collection:", mongo_collection)
print("Modo de carga:", modo_carga)

In [0]:
mongo_uri = dbutils.secrets.get(
    scope="mongodb",
    key="uri"
)

In [0]:
if not os.path.exists(arquivo_entrada):
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo_entrada}")

df = pd.read_excel(
    arquivo_entrada,
    sheet_name="chamados_raw"
)

print("Planilha carregada com sucesso!")
print("Linhas:", len(df))
print("Colunas:", len(df.columns))

display(df.head(10))

In [0]:
colunas_obrigatorias = [
    "id_chamado",
    "cliente_nome",
    "cliente_segmento",
    "cidade",
    "uf",
    "categoria",
    "prioridade",
    "status",
    "data_abertura",
    "data_fechamento",
    "canal_origem",
    "atendente",
    "descricao",
    "qtd_interacoes",
    "satisfacao",
    "sla_horas"
]

colunas_faltando = [
    coluna for coluna in colunas_obrigatorias
    if coluna not in df.columns
]

if colunas_faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {colunas_faltando}")

print("Validação de colunas concluída com sucesso!")

In [0]:
def limpar_valor(valor):
    if pd.isna(valor):
        return None

    if isinstance(valor, pd.Timestamp):
        return valor.to_pydatetime()

    return valor


def limpar_inteiro(valor):
    valor = limpar_valor(valor)

    if valor is None:
        return None

    return int(valor)


def limpar_float(valor):
    valor = limpar_valor(valor)

    if valor is None:
        return None

    return float(valor)


def montar_eventos(data_abertura, data_fechamento, status):
    eventos = []

    if data_abertura is not None:
        eventos.append({
            "tipo": "abertura",
            "data": data_abertura
        })

    if status in ["Em atendimento", "Pendente", "Resolvido", "Fechado"]:
        eventos.append({
            "tipo": "analise",
            "data": data_abertura
        })

    if data_fechamento is not None:
        eventos.append({
            "tipo": "fechamento",
            "data": data_fechamento
        })

    return eventos


def linha_para_documento(linha):
    data_abertura = limpar_valor(linha["data_abertura"])
    data_fechamento = limpar_valor(linha["data_fechamento"])

    id_chamado = str(linha["id_chamado"])

    documento = {
        "_id": id_chamado,
        "id_chamado": id_chamado,
        "cliente": {
            "nome": limpar_valor(linha["cliente_nome"]),
            "segmento": limpar_valor(linha["cliente_segmento"]),
            "cidade": limpar_valor(linha["cidade"]),
            "uf": limpar_valor(linha["uf"])
        },
        "categoria": limpar_valor(linha["categoria"]),
        "prioridade": limpar_valor(linha["prioridade"]),
        "status": limpar_valor(linha["status"]),
        "data_abertura": data_abertura,
        "data_fechamento": data_fechamento,
        "canal_origem": limpar_valor(linha["canal_origem"]),
        "atendente": limpar_valor(linha["atendente"]),
        "descricao": limpar_valor(linha["descricao"]),
        "qtd_interacoes": limpar_inteiro(linha["qtd_interacoes"]),
        "satisfacao": limpar_float(linha["satisfacao"]),
        "sla_horas": limpar_inteiro(linha["sla_horas"]),
        "eventos": montar_eventos(
            data_abertura=data_abertura,
            data_fechamento=data_fechamento,
            status=limpar_valor(linha["status"])
        ),
        "origem": {
            "sistema": "databricks",
            "arquivo": arquivo_entrada,
            "data_carga": datetime.now()
        }
    }

    return documento

In [0]:
documentos = [
    linha_para_documento(linha)
    for _, linha in df.iterrows()
]

print("Quantidade de documentos gerados:", len(documentos))

print("Exemplo do primeiro documento:")
print(json.dumps(documentos[0], indent=2, ensure_ascii=False, default=str))

In [0]:
client = MongoClient(
    mongo_uri,
    server_api=ServerApi("1"),
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=60000
)

client.admin.command("ping")

print("Conexão com MongoDB Atlas realizada com sucesso!")

In [0]:
db = client[mongo_database]
collection = db[mongo_collection]

if modo_carga == "replace_collection":
    collection.delete_many({})
    print("Collection limpa antes da carga.")

operacoes = [
    ReplaceOne(
        {"_id": documento["_id"]},
        documento,
        upsert=True
    )
    for documento in documentos
]

resultado = collection.bulk_write(operacoes)

print("Carga concluída com sucesso!")
print("Matched:", resultado.matched_count)
print("Modified:", resultado.modified_count)
print("Upserted:", resultado.upserted_count)

In [0]:
total_documentos = collection.count_documents({})

print("Total de documentos na collection:", total_documentos)

exemplo = collection.find_one()

print("Exemplo de documento salvo:")
print(json.dumps(exemplo, indent=2, ensure_ascii=False, default=str))

In [0]:
collection.create_index("id_chamado", unique=True)
collection.create_index("status")
collection.create_index("prioridade")
collection.create_index("categoria")
collection.create_index("cliente.segmento")

print("Índices criados/verificados com sucesso!")

In [0]:
print("Chamados por status:")

pipeline_status = [
    {
        "$group": {
            "_id": "$status",
            "quantidade": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "quantidade": -1
        }
    }
]

for item in collection.aggregate(pipeline_status):
    print(item)